# Notebook 03 — Feature Engineering

**Purpose:**
Enrich the merged bid dataset with derived features and backfill missing values.

**Inputs** (`data/processed/`):
- `merged_bid_data.xlsx`
- `merged_financial_evaluation.xlsx`

**Also reads** (`data/raw/`) for backfilling:
- `coal_india_tenders_details.xlsx` — title backfill
- `eprocure_result_tenders_detail.xlsx` — title backfill
- `gem_sow_output.xlsx` — structured GEM Statement of Work data (drilling meterage, mineral, state)

**Outputs** (`data/processed/`):
- `final_bid_data.xlsx` — fully enriched bid table
- `final_FE.xlsx` — financial eval with `drilling_meterage` and `per_meter_rate`

**Features engineered:**

| Feature | Source (priority order) |
|---------|------------------------|
| `title_description` | Raw portal backfill (Coal India → eProcure) |
| `mineral_name` | 1. GEM SOW structured data → 2. Regex on title |
| `state_name` | 1. GEM SOW structured data → 2. Regex on title + dept_org |
| `drilling_meterage` | 1. GEM SOW structured data → 2. Regex on title |
| `winner` | L1-rank imputation where awarded bidder not published |
| `winning_price` | L1-rank imputation |
| `tender_value` | Filled from minimum price in financial_eval |
| `work_order` | Contract URL from GEM tech eval file |
| `per_meter_rate` | `price / drilling_meterage` added to financial_eval |

**Extraction priority rule:**
Structured data (GEM SOW) always takes priority over regex extraction from titles.
Regex only runs on rows still missing after the structured source has been applied.
This avoids overwriting a known value with a regex guess.

---

## 1. Imports & Setup

In [15]:
import sys
import re
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from config import RAW_FILES, PROCESSED_FILES, DATA_RAW
from src.extractors import extract_mineral, extract_state_from_title
from src.parsers import extract_drilling_meterage
from src.cleaning import clean_company_name

print('Setup complete.')

Setup complete.


## 2. Load Data

In [16]:
bid_data = pd.read_excel(PROCESSED_FILES['merged_bids'])
fe_data  = pd.read_excel(PROCESSED_FILES['merged_fe'])

bid_data['bid_start_date'] = pd.to_datetime(bid_data['bid_start_date'], errors='coerce')
bid_data['bid_end_date']   = pd.to_datetime(bid_data['bid_end_date'],   errors='coerce')
bid_data.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
fe_data.drop( columns=['Unnamed: 0'], inplace=True, errors='ignore')

# Defensive re-check: fe_data['seller'] should already be clean coming out of
# Notebook 02. We re-apply here because this column feeds the L1 winner
# imputation below -- any uncleaned name would write directly into bid_data['winner'].
fe_data['seller'] = fe_data['seller'].astype(str).str.strip().apply(clean_company_name)

print(f"bid_data : {bid_data.shape}")
print(f"fe_data  : {fe_data.shape}")

bid_data : (5689, 13)
fe_data  : (22519, 7)


In [17]:
# Guard: every downstream .map() / .loc[mask] keys on bid_number.
# A non-unique bid_number silently misaligns rows. Fail loudly here instead.
assert bid_data['bid_number'].is_unique, (
    "bid_number is not unique in merged_bid_data.xlsx -- re-run Notebook 02."
)
print("Guard passed: bid_number is unique.")

Guard passed: bid_number is unique.


## 3. Load GEM Statement of Work Data

The GEM SOW file contains structured, pre-extracted values for drilling meterage,
mineral commodity, and state — far more reliable than regex over free-text titles.
This is loaded here and used as the **first-priority** source for those three features.
Regex extraction (Sections 6–8) only runs on rows still missing after this source
has been applied.

In [19]:
gem_sow_path = DATA_RAW / 'gem_sow.xlsx'

if gem_sow_path.exists():
    gem_sow_raw = pd.read_excel(gem_sow_path)
    gem_sow = (
        gem_sow_raw[['Bid No', 'Total Drilling (m)', 'Mineral Commodity', 'State / Area']]
        .drop_duplicates(subset='Bid No')
        .set_index('Bid No')
    )
    print(f"GEM SOW loaded: {len(gem_sow)} rows")
    print(f"  drilling_meterage available : {gem_sow['Total Drilling (m)'].notna().sum()}")
    print(f"  mineral_name available      : {gem_sow['Mineral Commodity'].notna().sum()}")
    print(f"  state_name available        : {gem_sow['State / Area'].notna().sum()}")
else:
    gem_sow = None
    print("gem_sow.xlsx not found -- will rely entirely on regex extraction.")

GEM SOW loaded: 1681 rows
  drilling_meterage available : 179
  mineral_name available      : 132
  state_name available        : 1593


## 4. Backfill Missing Titles

Some Coal India and eProcure bids arrived without a title in the standardised extract.
Titles are backfilled first because Sections 6–8 (regex extraction) depend on them.

In [20]:
missing_title = bid_data[bid_data['title_description'].isna()]
print(f"Missing titles before backfill: {len(missing_title)}")

# Coal India backfill
coal_raw  = pd.read_excel(RAW_FILES['coal_india'])
title_map = (
    coal_raw[coal_raw['Bid Number'].isin(missing_title['bid_number'])]
    .drop_duplicates(subset='Bid Number')
    .set_index('Bid Number')['Title/Items']
)
bid_data['title_description'] = bid_data['title_description'].fillna(
    bid_data['bid_number'].map(title_map)
)

# eProcure backfill
eprocure_raw = pd.read_excel(RAW_FILES['eprocure'])
title_map    = (
    eprocure_raw
    .drop_duplicates(subset='Bid Number')
    .set_index('Bid Number')['Title/Items']
)
bid_data['title_description'] = bid_data['title_description'].fillna(
    bid_data['bid_number'].map(title_map)
)

print(f"Missing titles after  backfill: {bid_data['title_description'].isna().sum()}")

Missing titles before backfill: 6
Missing titles after  backfill: 0


## 5. Mineral Name Extraction

**Priority 1:** GEM SOW structured data (exact commodity name, highly reliable).
**Priority 2:** Keyword search against 15-mineral reference list in tender title.
Regex only runs on rows still missing after the SOW source has been applied.

In [21]:
# Priority 1: GEM SOW structured data
if gem_sow is not None:
    bid_data['mineral_name'] = bid_data['bid_number'].map(gem_sow['Mineral Commodity'])
    n_from_sow = bid_data['mineral_name'].notna().sum()
    print(f"Mineral from GEM SOW  : {n_from_sow}")
else:
    bid_data['mineral_name'] = np.nan
    n_from_sow = 0

# Priority 2: Regex on title for rows still missing
mask = bid_data['mineral_name'].isna()
bid_data.loc[mask, 'mineral_name'] = (
    bid_data.loc[mask, 'title_description'].apply(extract_mineral)
)

n_total = bid_data['mineral_name'].notna().sum()
print(f"Mineral from regex    : {n_total - n_from_sow}")
print(f"Total with mineral    : {n_total} / {len(bid_data)} ({n_total/len(bid_data)*100:.1f}%)")
print("\nMineral distribution:")
print(bid_data['mineral_name'].value_counts().to_string())

Mineral from GEM SOW  : 132
Mineral from regex    : 1404
Total with mineral    : 1536 / 5689 (27.0%)

Mineral distribution:
mineral_name
Coal                                                                   1064
Manganese                                                               132
Copper                                                                   89
Iron                                                                     63
Bauxite                                                                  50
Gypsum                                                                   13
Rock                                                                      8
Limestone                                                                 7
REE & RM                                                                  5
REE                                                                       5
Potash, Halite, Poly Halite                                               5
VANADIUM                   

## 6. State Name Extraction

**Priority 1:** GEM SOW structured data.
**Priority 2:** Keyword search across combined title + department_org string
(using both fields improves coverage vs title-only).

In [22]:
# Priority 1: GEM SOW structured data
if gem_sow is not None:
    bid_data['state_name'] = bid_data['bid_number'].map(gem_sow['State / Area'])
    n_from_sow = bid_data['state_name'].notna().sum()
    print(f"State from GEM SOW    : {n_from_sow}")
else:
    bid_data['state_name'] = np.nan
    n_from_sow = 0

# Priority 2: Keyword search on combined title + department_org
# Combining both fields gives better coverage than title alone --
# e.g. "Bhubaneswar Office" in dept_org yields Odisha even when title is generic.
mask = bid_data['state_name'].isna()
bid_data.loc[mask, 'state_name'] = (
    bid_data.loc[mask]
    .apply(
        lambda x: extract_state_from_title(
            f"{x['title_description']} {x['department_org']}"
        ),
        axis=1
    )
)

n_total = bid_data['state_name'].notna().sum()
print(f"State from text search: {n_total - n_from_sow}")
print(f"Total with state      : {n_total} / {len(bid_data)} ({n_total/len(bid_data)*100:.1f}%)")

State from GEM SOW    : 1593
State from text search: 226
Total with state      : 1819 / 5689 (32.0%)


## 7. Drilling Meterage Extraction

**Priority 1:** GEM SOW structured data (exact figure from the tender document).
**Priority 2:** 3-tier regex cascade on tender title (direct qty → BH×depth → generic).
See `src/parsers.py` for full documentation of the regex cascade.

In [23]:
# Priority 1: GEM SOW structured data
if gem_sow is not None:
    bid_data['drilling_meterage'] = bid_data['bid_number'].map(gem_sow['Total Drilling (m)'])
    n_from_sow = bid_data['drilling_meterage'].notna().sum()
    print(f"Meterage from GEM SOW : {n_from_sow}")
else:
    bid_data['drilling_meterage'] = np.nan
    n_from_sow = 0

# Priority 2: Regex on title for rows still missing
mask = bid_data['drilling_meterage'].isna()
bid_data.loc[mask, 'drilling_meterage'] = (
    bid_data.loc[mask, 'title_description'].apply(extract_drilling_meterage)
)

n_total = bid_data['drilling_meterage'].notna().sum()
valid   = bid_data['drilling_meterage'].dropna()
print(f"Meterage from regex   : {n_total - n_from_sow}")
print(f"Total with meterage   : {n_total} / {len(bid_data)} ({n_total/len(bid_data)*100:.1f}%)")
if len(valid) > 0:
    print(f"Meterage range        : {valid.min():,.0f} m  —  {valid.max():,.0f} m")

Meterage from GEM SOW : 179
Meterage from regex   : 539
Total with meterage   : 718 / 5689 (12.6%)
Meterage range        : 0 m  —  353,800 m


## 8. Winner Imputation from L1 Bidder

Some tenders have a known L1 bidder in financial_eval but no Awarded Bidder on the portal.
Where there is exactly one unambiguous L1 bidder, they are imputed as the winner.
`fe_data['seller']` was cleaned in Section 2, so imputed names arrive clean.

In [24]:
l1_data = fe_data[fe_data['rank_position'] == 'L1'].copy()

l1_counts   = l1_data.groupby('bid_number')['seller'].nunique()
unambiguous = l1_counts[l1_counts == 1].index

l1_ref = (
    l1_data[l1_data['bid_number'].isin(unambiguous)]
    .drop_duplicates(subset='bid_number')
    .set_index('bid_number')
)

missing_mask = bid_data['winner'].isna()
n_to_impute  = missing_mask.sum()

bid_data.loc[missing_mask, 'winner']        = bid_data.loc[missing_mask, 'bid_number'].map(l1_ref['seller'])
bid_data.loc[missing_mask, 'winning_price'] = bid_data.loc[missing_mask, 'bid_number'].map(l1_ref['price'])

n_still_missing = bid_data['winner'].isna().sum()
print(f"Tenders missing winner before imputation : {n_to_impute}")
print(f"Winners imputed from L1 bidder           : {n_to_impute - n_still_missing}")
print(f"Tenders still missing winner             : {n_still_missing}")

Tenders missing winner before imputation : 469
Winners imputed from L1 bidder           : 418
Tenders still missing winner             : 51


## 9. Tender Value Backfill

Where `tender_value` is still missing, fill from the minimum price in financial_eval
for that tender (a reasonable proxy for the estimated value).

In [25]:
min_price_map = fe_data.groupby('bid_number')['price'].min()

n_missing_tv = bid_data['tender_value'].isna().sum()
mask = bid_data['tender_value'].isna()
bid_data.loc[mask, 'tender_value'] = bid_data.loc[mask, 'bid_number'].map(min_price_map)

resolved = n_missing_tv - bid_data['tender_value'].isna().sum()
print(f"tender_value backfilled from financial_eval : {resolved}")
print(f"tender_value still missing                  : {bid_data['tender_value'].isna().sum()}")

tender_value backfilled from financial_eval : 912
tender_value still missing                  : 8


## 10. GEM Work Order URL

Maps contract URLs from the GEM tech eval file onto bid_data as `work_order`.
If the tech eval file is absent, the column is set to NaN and the pipeline continues.

In [26]:
tech_eval_path = DATA_RAW / 'gem_specific_tech_eval_output.xlsx'

if tech_eval_path.exists():
    tech_eval = pd.read_excel(tech_eval_path)
    bid_url   = tech_eval[['Bid No', 'View Contract URL']].drop_duplicates(subset='Bid No')
    url_map   = bid_url.set_index('Bid No')['View Contract URL']
    bid_data['work_order'] = bid_data['bid_number'].map(url_map)
    print(f"work_order URLs mapped: {bid_data['work_order'].notna().sum()}")
else:
    bid_data['work_order'] = np.nan
    print("gem_specific_tech_eval_output.xlsx not found -- work_order set to NaN.")

work_order URLs mapped: 3223


## 11. Add drilling_meterage & per_meter_rate to Financial Eval

Map `drilling_meterage` from bid_data onto financial_eval keyed on bid_number,
then compute `per_meter_rate = price / drilling_meterage` (INR per metre).

In [27]:
meterage_map = bid_data.set_index('bid_number')['drilling_meterage']

fe_data['drilling_meterage'] = fe_data['bid_number'].map(meterage_map)
fe_data['per_meter_rate']    = fe_data['price'] / fe_data['drilling_meterage']

n_with_rate = fe_data['per_meter_rate'].notna().sum()
print(f"FE rows with per_meter_rate populated: {n_with_rate} / {len(fe_data)}")

FE rows with per_meter_rate populated: 2205 / 22519


## 12. Data Quality Summary

In [30]:
quality = pd.DataFrame({
    'missing_count'   : bid_data.isna().sum(),
    'completeness_pct': (bid_data.notna().mean() * 100).round(1),
    'dtype'           : bid_data.dtypes,
}).sort_values('completeness_pct')

print("Data Quality Report — final bid_data")
print("=" * 50)
print(quality.to_string())
print(f"\nTotal rows: {len(bid_data)}")

Data Quality Report — final bid_data
                   missing_count  completeness_pct           dtype
drilling_meterage           4971              12.6         float64
mineral_name                4153              27.0          object
state_name                  3870              32.0          object
contract_period             2514              55.8         float64
work_order                  2466              56.7          object
award_status                 187              96.7          object
winner                        51              99.1          object
winning_price                 51              99.1         float64
tender_value                   8              99.9         float64
bid_number                     0             100.0          object
title_description              0             100.0          object
ministry                       0             100.0          object
department_org                 0             100.0          object
bid_start_date           

## 13. Export

In [31]:
bid_data_export = bid_data.copy()
bid_data_export.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
bid_data_export.set_index('bid_number', drop=True, inplace=True)
bid_data_export.to_excel(PROCESSED_FILES['featured_bids'])

fe_data.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
fe_data.to_excel(PROCESSED_FILES['merged_fe'], index=False)

print(f"Exported final_bid_data  ->  {PROCESSED_FILES['featured_bids'].name}  ({len(bid_data_export)} rows)")
print(f"Exported final_FE        ->  {PROCESSED_FILES['merged_fe'].name}  ({len(fe_data)} rows)")
print("\nNotebook 03 complete. Proceed to 04_seller_master.ipynb.")

Exported final_bid_data  ->  final_bid_data.xlsx  (5689 rows)
Exported final_FE        ->  merged_financial_evaluation.xlsx  (22519 rows)

Notebook 03 complete. Proceed to 04_seller_master.ipynb.
